# Climate Hazards — All CA Carceral Facilities

Joins all tract-level and facility-level climate hazard data to the 357-facility base dataset.

**Output:** `data/ca_climate_hazards.csv`

## Hazard layers joined

| Layer | Join type | Source file |
|---|---|---|
| Heat & AQI | tract centroid → `tract_geoid` | `data/heat_air_hazard.csv` |
| Flood | tract centroid → `tract_geoid` | `data/flood_hazard.csv` |
| Drought | tract centroid → `tract_geoid` | `data/drought_hazard.csv` |
| Wildfire FHSZ | point-in-polygon | `data_sources/hazards/wildfire/calfire_fhsz.geojson` |
| Wildland-Urban Interface | point-in-polygon | `data_sources/hazards/wildfire/Wildland_Urban_Interface.zip` |

**Note on drought:** Run `data_sources/hazards/drought/drought_hazard.ipynb` first to generate `data/drought_hazard.csv`.

**Not included in this file (CDCR state prisons only):**
- Indoor/outdoor heat model: `data/indoor_outdoor_heat_2025.csv`
- Heat activation days: `data_sources/hazards/heat/heat_activations_*.csv`
- Heat risk index: `data/CDCR_heat_risk_index.csv`

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Base facilities
fac = pd.read_csv('data/ca_facilities.csv')
print(f'Base facilities: {len(fac)}')
print(f'Types: {fac["type"].value_counts().to_dict()}')

# Normalize tract_geoid to zero-padded 11-digit string
fac['tract_str'] = fac['tract_geoid'].astype(str).str.split('.').str[0].str.zfill(11)

## 1. Tract-level hazards — heat, flood, drought

In [ ]:
# ── Heat & AQI ──────────────────────────────────────────────────────────────
heat = pd.read_csv('data/heat_air_hazard.csv', dtype={'GEOID': str})
heat['tract_str'] = heat['GEOID'].str.zfill(11)

df = fac.merge(
    heat.drop(columns='GEOID'),
    on='tract_str', how='left'
)
print(f'After heat join: {df["heat_hazard_idx_norm"].notna().sum()} of {len(df)} facilities matched')

# ── Flood ────────────────────────────────────────────────────────────────────
flood = pd.read_csv('data/flood_hazard.csv', dtype={'GEOID': str})
flood['tract_str'] = flood['GEOID'].str.zfill(11)

df = df.merge(
    flood.drop(columns='GEOID'),
    on='tract_str', how='left'
)
print(f'After flood join: {df["flood_hazard_idx_norm"].notna().sum()} of {len(df)} facilities matched')

# ── Drought ──────────────────────────────────────────────────────────────────
import os
if os.path.exists('data/drought_hazard.csv'):
    drought = pd.read_csv('data/drought_hazard.csv', dtype={'GEOID': str})
    drought['tract_str'] = drought['GEOID'].str.zfill(11)
    df = df.merge(
        drought.drop(columns='GEOID'),
        on='tract_str', how='left'
    )
    print(f'After drought join: {df["drought_hazard_idx_norm"].notna().sum()} of {len(df)} facilities matched')
else:
    print('WARNING: data/drought_hazard.csv not found — run data_sources/hazards/drought/drought_hazard.ipynb first')
    print('Drought columns will be absent from output.')

## 2. Wildfire FHSZ — point-in-polygon

Fire Hazard Severity Zone classification (SRA + LRA layers combined).
Facilities outside all classified zones receive blank values (not classified = lowest risk).
`NonWildland` zones in the FHSZ layer are treated as unclassified.

In [ ]:
# Build facility GeoDataFrame (WGS84)
fac_geo = gpd.GeoDataFrame(
    df[['facilityid', 'longitude', 'latitude']].copy(),
    geometry=[Point(xy) for xy in zip(df['longitude'], df['latitude'])],
    crs='EPSG:4326'
)

# Load FHSZ layer
fhsz = gpd.read_file('data_sources/hazards/wildfire/calfire_fhsz.geojson')
if fhsz.crs is None:
    fhsz = fhsz.set_crs('EPSG:4326')
fhsz = fhsz.to_crs('EPSG:4326')

# Point-in-polygon
fac_fhsz = gpd.sjoin(
    fac_geo, fhsz[['fhsz', 'responsibility', 'geometry']],
    how='left', predicate='within'
)

# Handle multiple matches (keep highest severity if a point lands in overlapping zones)
severity_order = {'Very High': 3, 'High': 2, 'Moderate': 1, 'NonWildland': 0}
fac_fhsz['_sev'] = fac_fhsz['fhsz'].map(severity_order).fillna(-1)
fac_fhsz = (
    fac_fhsz.sort_values('_sev', ascending=False)
    .drop_duplicates('facilityid')
    .drop(columns=['_sev', 'index_right'])
)

# Replace NonWildland with blank (not classified)
fac_fhsz['fhsz'] = fac_fhsz['fhsz'].replace('NonWildland', '')
fac_fhsz = fac_fhsz.rename(columns={'responsibility': 'fhsz_responsibility'})

# Clear responsibility for unclassified facilities
fac_fhsz.loc[fac_fhsz['fhsz'] == '', 'fhsz_responsibility'] = ''

df = df.merge(
    fac_fhsz[['facilityid', 'fhsz', 'fhsz_responsibility']],
    on='facilityid', how='left'
)
df['fhsz'] = df['fhsz'].fillna('')
df['fhsz_responsibility'] = df['fhsz_responsibility'].fillna('')

print(f'FHSZ classification summary:')
print(df['fhsz'].value_counts(dropna=False).to_string())
print(f'\nResponsibility area summary:')
print(df['fhsz_responsibility'].value_counts(dropna=False).to_string())

## 3. Wildland-Urban Interface — point-in-polygon

WUI type from CalFire WUI boundaries (EPSG:3310). Facilities outside all WUI boundaries receive a blank value.

In [ ]:
# Load WUI shapefile (EPSG:3310) and reproject to WGS84
wui = gpd.read_file('data_sources/hazards/wildfire/Wildland_Urban_Interface.zip')
wui = wui.to_crs('EPSG:4326')

# Reproject facility points to match
fac_geo_wgs = fac_geo.to_crs('EPSG:4326')

# Point-in-polygon
fac_wui = gpd.sjoin(
    fac_geo_wgs, wui[['WUI_DESC', 'geometry']],
    how='left', predicate='within'
)

# Keep one row per facility — WUI_DESC priority: Intermix > Interface > Influence Zone
wui_order = {'Intermix': 3, 'Interface': 2, 'Influence Zone': 1}
fac_wui['_wui_rank'] = fac_wui['WUI_DESC'].map(wui_order).fillna(0)
fac_wui = (
    fac_wui.sort_values('_wui_rank', ascending=False)
    .drop_duplicates('facilityid')
    .drop(columns=['_wui_rank', 'index_right'])
)

fac_wui = fac_wui.rename(columns={'WUI_DESC': 'wui_type'})

df = df.merge(
    fac_wui[['facilityid', 'wui_type']],
    on='facilityid', how='left'
)
df['wui_type'] = df['wui_type'].fillna('')

print('WUI type summary:')
print(df['wui_type'].value_counts(dropna=False).to_string())

## 4. Output

In [ ]:
# Drop the working join key; keep everything else
output = df.drop(columns=['tract_str'])

# Column order: facility ID/meta, then hazard layers in a logical sequence
hazard_cols_heat = [
    'days_over_90_historic', 'days_over_90_midcentury', 'delta_90',
    'hotnights_pre_pct', 'hotnights_fut_pct',
    'PollutionP', 'AQI_norm',
    'heat_hazard_idx_norm', 'heat_hazard_fut_idx_norm',
]
hazard_cols_flood = [
    'flood_bam_100_pct', 'flood_bam_500_pct',
    'flood_verywet_pre_pct', 'flood_verywet_fut_pct',
    'flood_hazard_idx_norm', 'flood_hazard_fut_idx_norm',
]
hazard_cols_drought = [c for c in [
    'Dr_delta_JA_max_pre', 'Dr_delta_JA_max_fut',
    'Dr_WSV_average', 'Dr_precip_demand_ratio',
    'drought_delta_spei12_midcentury',
    'drought_hazard_idx_norm', 'drought_hazard_fut_idx_norm',
] if c in output.columns]
hazard_cols_fire = ['fhsz', 'fhsz_responsibility', 'wui_type']

facility_cols = [c for c in output.columns
                 if c not in hazard_cols_heat + hazard_cols_flood + hazard_cols_drought + hazard_cols_fire]
output = output[facility_cols + hazard_cols_heat + hazard_cols_flood + hazard_cols_drought + hazard_cols_fire]

output.to_csv('data/ca_climate_hazards.csv', index=False)
print(f'Saved {len(output)} rows × {len(output.columns)} columns to data/ca_climate_hazards.csv')
print(f'\nHazard column coverage (non-null):')
check_cols = ['heat_hazard_idx_norm', 'flood_hazard_idx_norm', 'fhsz']
if 'drought_hazard_idx_norm' in output.columns:
    check_cols.insert(2, 'drought_hazard_idx_norm')
for col in check_cols:
    n = output[col].replace('', float('nan')).notna().sum()
    print(f'  {col}: {n} of {len(output)}')